# Modeling V2 — TF-IDF baselines trên Gold V2

Chạy **hai model mới** (không dùng model V1 đã huấn luyện) trên **Gold V2**:

1. TF-IDF (1–2 grams) + multinomial **Logistic Regression**.
2. TF-IDF (1–2 grams) + **LinearSVC**.

Notebook dùng chung `training/factory.py`, `training/runner.py` và CLI `scripts/06_train_baseline.py` của repo. Chỉ thay nguồn Gold thành **V2**, lưu riêng tại `experiments/tfidf_baselines_v2/`; không sửa artifact V1. Mặc định `RUN_BALANCED=False` và `RUN_DIAGNOSTICS=False`.

**Chạy theo thứ tự:** Git checkout → cài đặt → mount Drive → verify Gold V2 → kiểm tra artifact hiện có → fit tuần tự → bảng Macro-F1/Weighted-F1/Accuracy trên **validation** → so sánh V1/V2 mô tả (nếu tương thích) → diagnostics tùy chọn → thử dự đoán Logistic Regression.

**Không fit TF-IDF trên validation/test; không dùng Gold test để chọn mô hình hay xem điểm.** `GoldReader.verify()` đọc file test chỉ để kiểm tra checksum. Phải tạo Gold V2 bằng notebook riêng trước khi chạy. Khi có nhiều Gold V2 manifests, chọn chính xác tên file trong cell Drive. Để thử nhánh chưa merge, đặt `VTC_GIT_REF=feat/modeling-v2-tfidf-colab` trong Colab trước khi chạy cell checkout (nhánh mặc định là `main`).

**Lưu ý khoa học:** V1/V2 Gold có thể khác cả text và train/validation split. Chênh lệch Macro-F1 giữa hai phiên bản chỉ là mô tả, **không phải bằng chứng riêng của bước làm sạch**. Source domain labels chưa được kiểm duyệt thủ công.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = __import__("os").environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_modeling_v2_project")

if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    )
    if dirty.stdout.strip():
        raise RuntimeError("Repository Colab có thay đổi chưa commit; hãy kiểm tra trước khi checkout.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} đã tồn tại nhưng không phải Git repository.")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Repository: {BRANCH} @ {revision}")


In [ ]:
import importlib
import subprocess
import sys

# Chỉ cài dependency phục vụ modeling; không cần tải lại dataset.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[modeling]"],
    cwd=str(PROJECT_DIR), check=True,
)

SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy Python package: {SRC_DIR}")
sys.path[:] = [p for p in sys.path if p != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))

# Colab keeps stale imports across git checkouts: refresh the whole package tree.
PACKAGE = "vietnamese_topic_classification"
for name in sorted(
    [name for name in sys.modules if name == PACKAGE or name.startswith(PACKAGE + ".")],
    key=len, reverse=True,
):
    del sys.modules[name]
importlib.invalidate_caches()

import sklearn
from vietnamese_topic_classification.config import get_domain_labels
from vietnamese_topic_classification.data.gold import GoldReader

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("Số domain labels:", len(get_domain_labels()))
assert len(get_domain_labels()) == 25

from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
MANIFEST_DIR = DATA_LAKE_ROOT / "03_gold" / "v2" / "manifests"
OUTPUT_ROOT = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v2"

if not MANIFEST_DIR.is_dir():
    raise FileNotFoundError(
        f"Chưa tìm thấy Gold V2 manifests: {MANIFEST_DIR}. "
        "Hãy chạy colab_gold_v2_runner.ipynb và kiểm tra mount Drive trước."
    )

# Multiple Gold V2 snapshots: choose one EXACT manifest filename explicitly.
GOLD_MANIFEST_NAME = None
available_manifests = sorted(p.name for p in MANIFEST_DIR.glob("gold_manifest_v2_*.json"))
print("Gold V2 manifests:", available_manifests)
if GOLD_MANIFEST_NAME is None:
    if len(available_manifests) != 1:
        raise ValueError(
            "Phải chọn GOLD_MANIFEST_NAME chính xác khi có 0 hoặc nhiều manifest. "
            f"Hiện có {len(available_manifests)}: {available_manifests}"
        )
    GOLD_MANIFEST_NAME = available_manifests[0]
if (
    Path(GOLD_MANIFEST_NAME).name != GOLD_MANIFEST_NAME
    or not GOLD_MANIFEST_NAME.startswith("gold_manifest_v2_")
    or not GOLD_MANIFEST_NAME.endswith(".json")
):
    raise ValueError("Chỉ chấp nhận tên file gold_manifest_v2_*.json.")

reader = GoldReader(DATA_LAKE_ROOT, GOLD_MANIFEST_NAME)
verified = reader.verify()  # SHA256/row count cả 3 split; không train từ test.
if reader.version != "v2" or reader.manifest.get("pipeline_version") != "v2":
    raise AssertionError("Gold manifest không phải V2.")
if reader.manifest["label_to_id"] != {
    label: idx for idx, label in enumerate(get_domain_labels())
}:
    raise AssertionError("Gold V2 label mapping không khớp canonical 25 labels.")
source = reader.manifest.get("source", {})
if (
    source.get("silver_version") != "v2"
    or not source.get("silver_v1_sha256")
    or not source.get("silver_report_sha256")
):
    raise AssertionError("Gold V2 thiếu lineage Silver V2 / Silver V1.")
if verified["split_rows"]["train"] < 2 or verified["split_rows"]["validation"] < 1:
    raise ValueError("Gold V2 train/validation không đủ để train.")

print("GOLD V2 VERIFIED:", verified)
print("Manifest:", reader.manifest_path)
print("Experiment output:", OUTPUT_ROOT)


In [ ]:
import hashlib
import json
import os

SEED = 42
MAX_FEATURES = 100_000
MIN_DF = 2
RUN_BALANCED = False  # Optional LR balanced experiment.
RUN_DIAGNOSTICS = False  # Optional, costly refits for charts.

# Hai baseline mặc định: LogisticRegression và LinearSVC, cùng TF-IDF.
EXPERIMENT_SPECS = [
    {"slug": "tfidf_logreg", "classifier": "logistic_regression", "class_weight": "none"},
    {"slug": "tfidf_linear_svc", "classifier": "linear_svc", "class_weight": "none"},
]
if RUN_BALANCED:
    EXPERIMENT_SPECS.append(
        {"slug": "tfidf_logreg_balanced",
         "classifier": "logistic_regression", "class_weight": "balanced"}
    )

if MAX_FEATURES < 1 or MIN_DF < 1:
    raise ValueError("MAX_FEATURES và MIN_DF phải dương.")

manifest_sha256 = hashlib.sha256(reader.manifest_path.read_bytes()).hexdigest()
for spec in EXPERIMENT_SPECS:
    spec["name"] = f"{spec['slug']}_v2_{manifest_sha256[:10]}"
    spec["dir"] = OUTPUT_ROOT / spec["name"]

def verify_existing_experiment(spec):
    """Chỉ skip khi model + reports hoàn chỉnh, cùng Gold source và hyperparams."""
    directory = spec["dir"]
    expected_files = ("model.joblib", "config.json", "metadata.json", "validation_metrics.json")
    missing = [name for name in expected_files if not (directory / name).is_file()]
    if missing:
        raise FileExistsError(f"Experiment V2 chưa hoàn chỉnh: thiếu {missing}: {directory}")
    if (directory / "model.joblib").stat().st_size == 0:
        raise ValueError(f"Model rỗng: {directory / 'model.joblib'}")
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    report = json.loads((directory / "validation_metrics.json").read_text(encoding="utf-8"))
    expected_weight = None if spec["class_weight"] == "none" else "balanced"
    if (
        config.get("name") != spec["name"]
        or metadata.get("experiment_name") != spec["name"]
        or metadata.get("test_evaluated") is not False
        or metadata.get("lineage", {}).get("gold_manifest_name") != GOLD_MANIFEST_NAME
        or metadata.get("lineage", {}).get("gold_source") != source
        or config.get("gold_manifest_name") != GOLD_MANIFEST_NAME
        or config.get("seed") != SEED
        or config.get("classifier", {}).get("kind") != spec["classifier"]
        or config.get("classifier", {}).get("params", {}).get("class_weight") != expected_weight
        or config.get("features", {}).get("kind") != "tfidf"
        or config.get("features", {}).get("params", {}).get("max_features") != MAX_FEATURES
        or config.get("features", {}).get("params", {}).get("min_df") != MIN_DF
        or config.get("features", {}).get("params", {}).get("ngram_range") != [1, 2]
        or metadata.get("lineage", {}).get("gold_manifest_sha256") != manifest_sha256
        or any(not isinstance(report.get(metric), (int, float))
               for metric in ("macro_f1", "weighted_f1", "accuracy"))
        or len(report.get("per_class_f1", {})) != 25
        or len(report.get("confusion_matrix", [])) != 25
    ):
        raise ValueError(f"Artifact không khớp cấu hình/Gold hiện tại: {directory}")
    return report

# Preflight toàn bộ trước khi bắt đầu để không tự ghi đè artifacts cũ.
existing_reports = {}
for spec in EXPERIMENT_SPECS:
    if spec["dir"].exists():
        existing_reports[spec["name"]] = verify_existing_experiment(spec)
        print("VERIFIED EXISTING — SKIP:", spec["dir"])
    else:
        print("READY TO TRAIN:", spec["name"])
print("Tổng số baseline:", len(EXPERIMENT_SPECS))


In [ ]:
# Gọi training CLI gốc, tránh duplicate sklearn fit/metric logic trong notebook.
# Chạy TUẦN TỰ để hạn chế RAM trên Colab CPU.
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    part for part in [str(SRC_DIR), env.get("PYTHONPATH", "")] if part
)

for spec in EXPERIMENT_SPECS:
    if spec["name"] in existing_reports:
        print("SKIPPED (verified):", spec["name"])
        continue
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/06_train_baseline.py"),
        "--manifest-name", GOLD_MANIFEST_NAME,
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--output-root", str(OUTPUT_ROOT),
        "--name", spec["name"],
        "--classifier", spec["classifier"],
        "--class-weight", spec["class_weight"],
        "--max-features", str(MAX_FEATURES),
        "--min-df", str(MIN_DF),
        "--seed", str(SEED),
    ]
    print("TRAINING:", spec["name"], "(", spec["classifier"], ")")
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    existing_reports[spec["name"]] = verify_existing_experiment(spec)
    print("SAVED:", spec["dir"])
print("ALL BASELINE EXPERIMENTS READY")


In [ ]:
# V2 validation ONLY — chưa sử dụng test để chọn mô hình.
from IPython.display import Markdown, display

rows = []
for spec in EXPERIMENT_SPECS:
    metrics = verify_existing_experiment(spec)
    if len(metrics.get("per_class_f1", {})) != 25:
        raise AssertionError(f"Thiếu per-class F1 (25 nhãn): {spec['name']}")
    if len(metrics.get("confusion_matrix", [])) != 25:
        raise AssertionError(f"Confusion matrix không đủ 25 hàng: {spec['name']}")
    rows.append({
        "name": spec["name"],
        "classifier": spec["classifier"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "artifact_dir": str(spec["dir"]),
    })

rows.sort(key=lambda row: (-row["macro_f1"], row["name"]))
headers = "| Rank | Model | Macro-F1 | Weighted-F1 | Accuracy |"
divider = "|---:|---|---:|---:|---:|"
lines = [
    f"| {idx} | {row['name']} | {row['macro_f1']:.4f} "
    f"| {row['weighted_f1']:.4f} | {row['accuracy']:.4f} |"
    for idx, row in enumerate(rows, 1)
]
display(Markdown("\n".join([headers, divider, *lines])))
print("Best on VALIDATION Macro-F1:", rows[0]["name"])
print("Selected artifact:", rows[0]["artifact_dir"])
print("Test set CHƯA được đánh giá.")


In [ ]:
# Read-only, descriptive comparison to V1: same classifier/TF-IDF settings.
# If V1 has multiple compatible experiments, SKIP instead of silently choosing one.
from IPython.display import Markdown, display

V1_ROOT = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v1"
comparison = []
for spec in EXPERIMENT_SPECS:
    matches = []
    for metrics_file in sorted(V1_ROOT.glob(f"{spec['slug']}_v1_*/validation_metrics.json")):
        experiment_dir = metrics_file.parent
        config_file = experiment_dir / "config.json"
        metadata_file = experiment_dir / "metadata.json"
        if not (config_file.is_file() and metadata_file.is_file()):
            continue
        old_config = json.loads(config_file.read_text(encoding="utf-8"))
        old_meta = json.loads(metadata_file.read_text(encoding="utf-8"))
        old_metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
        expected_weight = None if spec["class_weight"] == "none" else "balanced"
        if (
            old_config.get("classifier", {}).get("kind") != spec["classifier"]
            or old_config.get("classifier", {}).get("params", {}).get("class_weight") != expected_weight
            or old_config.get("features", {}).get("kind") != "tfidf"
            or old_config.get("features", {}).get("params", {}).get("max_features") != MAX_FEATURES
            or old_config.get("features", {}).get("params", {}).get("min_df") != MIN_DF
            or old_config.get("features", {}).get("params", {}).get("ngram_range") != [1, 2]
            or old_config.get("seed") != SEED
            or old_meta.get("test_evaluated") is not False
            or old_meta.get("lineage", {}).get("gold_source", {}).get("silver_version") not in (None, "v1")
            or not isinstance(old_metrics.get("macro_f1"), (int, float))
        ):
            continue
        matches.append((experiment_dir.name, old_metrics["macro_f1"]))
    if len(matches) != 1:
        print(f"{spec['slug']}: tìm thấy {len(matches)} V1 baseline tương thích; "
              "bỏ qua để tránh chọn sai snapshot.")
        continue
    current = verify_existing_experiment(spec)
    comparison.append((spec["slug"], matches[0][0], matches[0][1],
                       spec["name"], current["macro_f1"]))
if comparison:
    lines = [
        "| Classifier | V1 validation Macro-F1 | V2 validation Macro-F1 | Delta (V2−V1) |",
        "|---|---:|---:|---:|",
    ]
    for slug, name_v1, score_v1, name_v2, score_v2 in comparison:
        lines.append(f"| {slug} | {score_v1:.4f} | {score_v2:.4f} | {score_v2-score_v1:+.4f} |")
        print(f"Compared: {name_v1} → {name_v2}")
    display(Markdown("\n".join(lines)))
else:
    print("Không có V1 baseline duy nhất, tương thích để so sánh.")
print("CẢNH BÁO: V1/V2 có thể khác split và text. Delta không đo riêng tác động cleaning.")


In [ ]:
# Optional post-fit charts. The existing diagnostics CLI calls its output directory "diagnostics_v1"
# because that describes the diagnostics FORMAT, not the dataset version.
if RUN_DIAGNOSTICS:
    # Hậu kiểm cho các experiment đã train: chạy lại chỉ cell này nếu đã có model.
    # Mỗi experiment ghi 6 ảnh PNG + diagnostics.json dưới cùng thư mục Drive.
    import subprocess
    
    for spec in EXPERIMENT_SPECS:
        command = [
            sys.executable, str(PROJECT_DIR / "scripts/08_plot_diagnostics.py"),
            "--data-lake-root", str(DATA_LAKE_ROOT),
            "--artifact-dir", str(spec["dir"]),
        ]
        print("DIAGNOSTICS:", spec["name"], flush=True)
        subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
        chart_dir = spec["dir"] / "diagnostics_v1"
        for name in (
            "train_vs_validation.png",
            "learning_curve_by_train_size.png",
            "correct_counts_by_class.png",
            "recall_by_class.png",
            "per_class_f1.png",
            "confusion_matrix_normalized.png",
            "diagnostics.json",
        ):
            if not (chart_dir / name).is_file():
                raise FileNotFoundError(chart_dir / name)
        report = json.loads((chart_dir / "diagnostics.json").read_text(encoding="utf-8"))
        print("Solver iterations (not epochs):", report["solver_iterations"])
        print("Train minus validation gap:", report["train_minus_validation_gap"])
        print("Saved charts:", chart_dir)
    
else:
    print("SKIPPED diagnostics: RUN_DIAGNOSTICS=False")


In [ ]:
# Demo xác suất Softmax chỉ cho Logistic Regression.
# Chỉ load artifact do chính notebook này tạo/đã kiểm chứng; joblib là định dạng pickle.
import joblib

logreg_spec = next(
    spec for spec in EXPERIMENT_SPECS if spec["classifier"] == "logistic_regression"
)
pipeline = joblib.load(logreg_spec["dir"] / "model.joblib")
example_text = "Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận đấu."
probabilities = pipeline.predict_proba([example_text])[0]
labels = get_domain_labels()
ranked = sorted(
    ((labels[int(label_id)], float(probabilities[idx]))
     for idx, label_id in enumerate(pipeline.classes_)),
    key=lambda item: item[1], reverse=True,
)
print("Ví dụ:", example_text)
print("Top 5 dự đoán (model Logistic Regression):")
for domain, probability in ranked[:5]:
    print(f"{domain:30s}: {probability:.3%}")
print("Tổng xác suất:", sum(probabilities))
# LinearSVC không có predict_proba mặc định: decision scores KHÔNG là xác suất.


## Kết quả Modeling V2

- Gold source: `03_gold/v2/manifests/<gold_manifest_v2_...>.json` và immutable train/validation/test.
- Outputs: `experiments/tfidf_baselines_v2/<experiment_name>/` chứa `model.joblib`, `config.json`, `metadata.json`, `validation_metrics.json`.
- Hai baseline mặc định là TF-IDF + Logistic Regression và TF-IDF + LinearSVC. `RUN_BALANCED=True` bổ sung một LR balanced, không sửa hai model đã có.
- `RUN_DIAGNOSTICS=True` bật chart hậu kiểm (có thể mất nhiều CPU/RAM); tên `diagnostics_v1/` là phiên bản **định dạng diagnostics**, không phải mô hình V1.
- Nếu điều chỉnh `MAX_FEATURES`, `MIN_DF`, `SEED`, v.v., **đổi slug experiment** trước khi train; không ghi đè artifacts cũ. Notebook kiểm tra cấu hình nguồn rồi mới quyết định skip.
- Không đánh giá test trong notebook. Chọn model trên validation, khóa cấu hình rồi mới đánh giá test bằng workflow riêng.
- So sánh V1/V2 chỉ mang tính mô tả vì split và số mẫu có thể khác. Cần một **benchmark/test cố định** để đo tác động riêng của cleaning.

25 nhãn VTSNLP là nhãn nguồn tự động, chưa được kiểm duyệt thủ công.
